In [46]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    classification_report,
    balanced_accuracy_score,
    confusion_matrix,
)

In [47]:
X = pd.read_parquet('data/processed/baseline_0_features.parquet')
y = pd.read_parquet('data/processed/target.parquet')

In [48]:
print(X.shape)
print(y.shape)

(51055, 384)
(51055, 7)


In [49]:
y.columns

Index(['Anxiety', 'Bipolar', 'Depression', 'Normal', 'Personality disorder',
       'Stress', 'Suicidal'],
      dtype='str')

In [60]:
y.sum(axis=0)

Anxiety                  3617
Bipolar                  2501
Depression              15078
Normal                  16037
Personality disorder      895
Stress                   2293
Suicidal                10634
dtype: int64

In [50]:
y_labels = np.argmax(y, axis=1)
y_labels

array([0, 0, 0, ..., 0, 0, 0], shape=(51055,))

In [51]:
# stratified train-test split to preserve class distribution
X_train, X_test, y_train, y_test = train_test_split(X, y_labels, test_size=0.2, random_state=42, stratify=y_labels)

### Naive Model

In [63]:
pd.DataFrame(y_train)[0].value_counts() # so always predict class 3 (normal) baseline evaluation

0
3    12830
2    12062
6     8507
0     2894
1     2001
5     1834
4      716
Name: count, dtype: int64

In [67]:
y_pred_naive = np.array([3 for _ in range(y_test.shape[0])])

In [68]:
print("=== Naive Model ===")
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred_naive))
print(classification_report(y_test, y_pred_naive, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_naive))

=== Naive Model ===
Balanced Accuracy: 0.14285714285714285
              precision    recall  f1-score   support

           0      0.000     0.000     0.000       723
           1      0.000     0.000     0.000       500
           2      0.000     0.000     0.000      3016
           3      0.314     1.000     0.478      3207
           4      0.000     0.000     0.000       179
           5      0.000     0.000     0.000       459
           6      0.000     0.000     0.000      2127

    accuracy                          0.314     10211
   macro avg      0.045     0.143     0.068     10211
weighted avg      0.099     0.314     0.150     10211

Confusion Matrix:
 [[   0    0    0  723    0    0    0]
 [   0    0    0  500    0    0    0]
 [   0    0    0 3016    0    0    0]
 [   0    0    0 3207    0    0    0]
 [   0    0    0  179    0    0    0]
 [   0    0    0  459    0    0    0]
 [   0    0    0 2127    0    0    0]]


c:\Users\alfre\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\alfre\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\alfre\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

### Logistic Regression

In [54]:
# Logistic Regression with class_weight to handle imbalance
log_reg = LogisticRegression(max_iter=1000, class_weight="balanced")
log_reg.fit(X_train, y_train)
y_pred_lr = log_reg.predict(X_test)

In [55]:
# Evaluation metrics
print("=== Logistic Regression ===")
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))

=== Logistic Regression ===
Balanced Accuracy: 0.7354870717681813
              precision    recall  f1-score   support

           0      0.719     0.795     0.755       723
           1      0.627     0.768     0.691       500
           2      0.790     0.563     0.657      3016
           3      0.919     0.889     0.903      3207
           4      0.258     0.721     0.380       179
           5      0.415     0.715     0.525       459
           6      0.658     0.698     0.677      2127

    accuracy                          0.730     10211
   macro avg      0.627     0.735     0.656     10211
weighted avg      0.764     0.730     0.737     10211

Confusion Matrix:
 [[ 575   30   15   23   18   55    7]
 [  27  384   20    3   29   24   13]
 [  93  130 1698   95  171  148  681]
 [  51   10   42 2850   48  160   46]
 [   6    7    7    4  129   14   12]
 [  33   20   22   15   28  328   13]
 [  15   31  346  112   77   61 1485]]


### Decision Tree

In [56]:
# Decision Tree with class_weight to handle imbalance
dt = DecisionTreeClassifier(class_weight="balanced", random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

In [57]:
print("\n=== Decision Tree ===")
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))


=== Decision Tree ===
Balanced Accuracy: 0.39988744669213677
              precision    recall  f1-score   support

           0      0.446     0.438     0.442       723
           1      0.388     0.390     0.389       500
           2      0.480     0.484     0.482      3016
           3      0.746     0.720     0.733      3207
           4      0.159     0.156     0.158       179
           5      0.192     0.200     0.196       459
           6      0.394     0.409     0.402      2127

    accuracy                          0.516     10211
   macro avg      0.401     0.400     0.400     10211
weighted avg      0.520     0.516     0.518     10211

Confusion Matrix:
 [[ 317   39  143   89   13   50   72]
 [  33  195  130   44   15   17   66]
 [ 125  140 1461  295   51  134  810]
 [ 119   39  333 2309   28   98  281]
 [   6    8   69   22   28   13   33]
 [  47   17  137   78   13   92   75]
 [  63   65  769  257   28   74  871]]
